# धडा 17 - Foundry Local आणि Qwen सह स्थानिक AI एजंट तयार करणे

या नोटबुकमध्ये आपण एक **स्थानिक अभियांत्रिकी सहाय्यक** तयार करता जो पूर्णपणे आपल्या वर्कस्टेशनवर चालतो. कोणतीही क्लाउड इनफरन्स कधीही वापरली जात नाही. सहाय्यक खालील गोष्टी करू शकतो:

1. **Foundry Local द्वारे Qwen फंक्शन कॉलिंगद्वारे साधने कॉल करा.**
2. **सँडबॉक्स्ड प्रोजेक्ट निर्देशिकेतील फाईल्स लिस्ट करा आणि वाचा.**
3. **साध्या मेट्रिक्ससह कोडचे विश्लेषण करा.**
4. **स्थानिक RAG (Chroma) सह दस्तऐवजीकरण शोधा.**
5. **स्थानिक MCP सर्व्हर वापरा** (जर कॉन्फिगर केले नसेल तर सुरेखपणे स्किप केले जाते).

एजंट कोड जवळजवळ क्लाउड धड्यासारखाच दिसतो — फक्त क्लायंट एंडपॉइंट क्लाउडमधून `localhost` कडे हलतो.


## सेटअप

हा नोटबुक चालवण्यापूर्वी:

1. **Microsoft Foundry Local इंस्टॉल करा** (आपल्या OS साठी [डॉक्युमेंटेशन](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) पहा).
2. **Qwen मॉडेल डाउनलोड करा आणि सुरु करा:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. खालील Python पॅकेजेस इंस्टॉल करा.

सर्व काही स्थानिकपणे चालते. सुमारे 8 GB RAM असलेली मशीन किमान वास्तविक आहे.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. फाउंड्री लोकलशी कनेक्ट करा

`FoundryLocalManager` जर आवश्यक असेल तर मॉडेल डाउनलोड करते, लोकल सेवा सुरू करते, आणि आम्हाला **OpenAI-सुसंगत एंडपॉइंट** देते. नंतर आपण सामान्य OpenAI SDK त्याकडे निर्देशित करतो. API की ही एक स्थानिक प्लेसहोल्डर आहे — कोणतीही क्लाउड क्रेडेन्शियल वापरली जात नाही.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. स्थानिक साधने (सँडबॉक्स केलेली फाइल ऑपरेशन्स)

आम्ही डिस्कवर एक लहान नमुना प्रकल्प तयार करतो, नंतर त्या प्रकल्पाच्या मूळ निर्देशिकेपर्यंत मर्यादित साधने परिभाषित करतो. सँडबॉक्स तपासणी स्थानिकरीत्या देखील महत्त्वाची असते: कोणतीही साधन जी मनमानी मार्ग वाचते ती तुमच्या वापरकर्त्याच्या परवानग्यांसह चालते आणि तुम्ही ज्या कोणत्याही गोष्टींना स्पर्श करू शकता तिला स्पर्श करू शकते.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. स्थानिक RAG सह Chroma

आम्ही दस्तऐवजाच्या लहान संचाच्या तुकड्यांना स्थानिक Chroma संग्रहात एम्बेड करतो. Chroma ऑन-प्रोसेस चालते आणि व्हेक्टर डिस्कवर संग्रहित करते — कोणताही सर्व्हर नाही, कोणताही क्लाउड नाही. `search_docs` साधन क्वेरीसाठी सगळ्यात संबंधित तुकडे परत आणते.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. टूल-कॉलिंग लूप

आता आपण OpenAI टूल्स स्कीमा वापरून मॉडेलसह टूल नोंदणी करतो आणि मानक टूल-कॉलिंग लूप चालवतो — मॉडेल टूलची विनंती करते, आम्ही ते स्थानिक पद्धतीने चालवतो, निकाल परत देतो, आणि हे प्रक्रिया तिथपर्यंत चालू ठेवतो जोपर्यंत मॉडेल अंतिम उत्तर तयार करत नाही. क्युएनचे विश्वसनीय फंक्शन कॉलिंग हेच कारण आहे की हे ऑन-डिव्हाइस कार्य करते.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. स्थानिक MCP (पर्यायी)

MCP हा एक ट्रान्सपोर्ट आहे, क्लाउड सेवा नाही — MCP सर्व्हर स्थानिक प्रक्रियेप्रमाणे `stdio` वर चालू शकतो. खालील सेल दाखवते की आपण स्थानिक MCP सर्व्हरशी कसा कनेक्ट करू शकता जर तो एक कॉन्फिगर केलेला असेल (उदाहरणार्थ, फाइलसिस्टम सर्व्हर). `LOCAL_MCP_COMMAND` सेट नसेल तर तो सौम्यपणे वळतो, त्यामुळे नोटबुक अजूनही अखंडपणे चालू राहतो.

सुरक्षा सूचना: स्थानिक MCP सर्व्हर आपल्या वापरकर्त्याच्या परवानग्यांसह चालतो. त्याला प्रकल्प निर्देशिकेपुरता मर्यादित करा आणि त्याच्या आउटपुटवर कार्यवाही करण्यापूर्वी त्याची पडताळणी करा.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## सारांश

आपण एक अभियांत्रिकी सहाय्यक तयार केला जो फक्त आपल्या मशीनवर चालतो:

- **Foundry Local** ने OpenAI-सुसंगत एंडपॉईंटच्या मागे **Qwen** मॉडेल दिले — जेणेकरून एजंट कोड क्लाउड धड्यांसोबत जुळतो.
- **Sandboxed tools** ने एजंटला फायलींमध्ये प्रवेश आणि कोड विश्लेषण दिले, प्रोजेक्ट डिरेक्टरी सोडवता येत नाही.
- **Chroma** ने कागदपत्रांवर **स्थानिक RAG** उपलब्ध करून दिला.
- **Local MCP** ने दाखवले की कसे MCP इकोसिस्टम ऑफलाइन पुन्हा वापरू शकतो.

कोणत्याही टप्प्यावर क्लाउड अनुमान वापरले गेले नाही.

### आव्हान

स्थानिक एजंटला विस्तार करा:

1. **अनेक MCP सर्व्हरांसह काम करा** — फाइलसिस्टम सर्व्हर आणि गिट सर्व्हर एकत्र कनेक्ट करा आणि एजंटला त्यांच्यात निवड करु द्या.
2. **स्थानिक मेमरी वापरा** — एक लहान संभाषण इतिहास डिस्कवर जतन करा जेणेकरून सहाय्यक नोंदणीक पुनरारंभांदरम्यान पूर्वीचे टप्पे लक्षात ठेवेल.
3. **स्थानिक मल्टी-एजंट समन्वयाला आधार द्या** — दुसरा स्थानिक एजंट जोडा (उदा. एक पुनरावलोकक) आणि दोन एजंट एकत्र काम करतील.

पुढील धड्यात आपण जाणून घ्याल की तैनात AI एजंट्सना कसे सुरक्षित करायचे.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
